# Задание 11. Обучение модели обнаружения опасных ссылок

**Цель:** обучить и сравнить три алгоритма классификации URL (`LogisticRegression`, `RandomForestClassifier`, `HistGradientBoostingClassifier`), подобрать пороги `t1` и `t2` для категорий `SAFE`, `SUSPICIOUS`, `DANGEROUS`, оценить лучшую модель на отложенной тестовой выборке и проверить на 20 искусственных URL.

In [1]:
import json
import pandas as pd

with open('../ml/artifacts/url_model_meta.json', 'r', encoding='utf-8') as f:
    meta = json.load(f)

comp_df = pd.DataFrame(meta['validation_comparison']).T.drop(columns=['confusion_matrix'])
print('Сравнение моделей на валидационной выборке:')
print(comp_df.to_string())

Сравнение моделей на валидационной выборке:
                     threshold accuracy precision  recall      f1      f2 roc_auc  pr_auc train_time_sec latency_ms_per_url
LogisticRegression         0.5   0.8349    0.8097  0.8147  0.8122  0.8137  0.8886  0.8895          2.747             1.4004
RandomForest               0.5   0.9419     0.932  0.9357  0.9339   0.935  0.9749  0.9725          7.614           106.0401
HistGradientBoosting       0.5   0.9497    0.9361    0.95   0.943  0.9472  0.9807  0.9797         10.727             23.964


## Выбор модели, пороги `t1` / `t2` и однократная оценка на тесте

- **Выбранная модель:** `HistGradientBoosting` с параметрами `{'learning_rate': 0.05, 'max_depth': 10, 'max_iter': 250}`
- **Пороги категоризации:** `t1 = 0.35` (`SAFE < t1 <= SUSPICIOUS < t2`), `t2 = 0.51` (`DANGEROUS >= t2`)
- **Метрики на тесте (`test`):** Accuracy = `0.9466`, Precision = `0.9479`, Recall = `0.9363`, F1 = `0.9421`, F2 = `0.9386`, ROC-AUC = `0.9788`

In [2]:
print('Метрики на валидации (при пороге t2):', meta['val_metrics'])
print('Метрики на тесте (при пороге t2):', meta['test_metrics'])
print('\nТоп-10 важнейших признаков (Random Forest):')
print(pd.DataFrame(meta['feature_importance']['random_forest_top10']).to_string(index=False))

Метрики на валидации (при пороге t2): {'threshold': 0.51, 'accuracy': 0.9499, 'precision': 0.9368, 'recall': 0.9497, 'f1': 0.9432, 'f2': 0.9471, 'roc_auc': 0.9807, 'pr_auc': 0.9797, 'train_time_sec': 10.727, 'latency_ms_per_url': 23.964, 'confusion_matrix': {'tn': 4341, 'fp': 228, 'fn': 179, 'tp': 3382}}
Метрики на тесте (при пороге t2): {'threshold': 0.51, 'accuracy': 0.9466, 'precision': 0.9479, 'recall': 0.9363, 'f1': 0.9421, 'f2': 0.9386, 'roc_auc': 0.9788, 'pr_auc': 0.9804, 'train_time_sec': 10.727, 'latency_ms_per_url': 23.964, 'confusion_matrix': {'tn': 4491, 'fp': 209, 'fn': 259, 'tp': 3805}}

Топ-10 важнейших признаков (Random Forest):
           feature  importance
       path_length     0.23079
          is_https     0.14620
    num_digits_url     0.12187
        url_length     0.08643
digit_ratio_domain     0.07857
     domain_length     0.07713
    domain_entropy     0.06054
    num_subdomains     0.05827
          num_dots     0.04112
num_hyphens_domain     0.03190


In [3]:
synth_df = pd.DataFrame(meta['synthetic_20_evaluation']['results'])
print(f"Проверка на 20 искусственных URL: {meta['synthetic_20_evaluation']['matched']}/{meta['synthetic_20_evaluation']['total']}")
print(synth_df.to_string(index=False))

Проверка на 20 искусственных URL: 20/20
                                            url expected_category  p_url predicted_category  match
                  https://www.example.com/about              SAFE 0.0128               SAFE   True
                               https://kaspi.kz              SAFE 0.0214               SAFE   True
                                https://egov.kz              SAFE 0.0214               SAFE   True
                  https://halykbank.kz/ru/cards              SAFE 0.0103               SAFE   True
         https://www.google.com/search?q=python              SAFE 0.0055               SAFE   True
                 https://kazpost.kz/ru/tracking              SAFE 0.0384               SAFE   True
                         https://jusan.kz/about              SAFE 0.0525               SAFE   True
                       https://forte.kz/credits              SAFE 0.0177               SAFE   True
                        https://bcc.kz/personal              SAFE 0.2